In [ ]:
!pip install scanpy

In [ ]:
#!/usr/bin/env python3
"""
PT-1/PT-2 re-annotation against Xu et al. 2024 / Kann et al. 2025
failed-repair proximal tubule markers.

=== COLAB VERSION ===
Mount Google Drive first, then run all cells.
"""

# ── Cell 1: Mount Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Cell 2: Install & import ─────────────────────────────────
!pip install scanpy anndata -q

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Cell 3: Paths ────────────────────────────────────────────
SN_DIR = "/content/drive/MyDrive/IMSB/projects/Hubers lab/Aging_DEG_Enrichment_Cellchat/files/sn"
KIDNEY_H5AD = f"{SN_DIR}/annotated-aging-soupxed.h5ad"
# MUSCLE_H5AD = f"{SN_DIR}/annotated-muscle-soupxed.h5ad"  # not needed for PT analysis

OUTPUT_DIR = "/content/drive/MyDrive/IMSB/projects/Hubers lab/Aging_DEG_Enrichment_Cellchat/thesis_supp_analysis/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

KIDNEY_CONDITION_MAP = {
    'sg18': 'ctrl', 'sg20': 'age', 'sg24': 'DR', 'sg26': 'sAct', 'sg28': 'combi'
}
cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']

# ── Cell 4: Load kidney ──────────────────────────────────────
print("Loading kidney h5ad (this may take a few minutes for 2.6GB)...")
k_adata = sc.read_h5ad(KIDNEY_H5AD)

# Map conditions
if 'sample' in k_adata.obs.columns:
    cond_col = 'sample'
elif 'orig.ident' in k_adata.obs.columns:
    cond_col = 'orig.ident'
else:
    cond_col = k_adata.obs.columns[0]
k_adata.obs['condition'] = k_adata.obs[cond_col].astype(str).map(KIDNEY_CONDITION_MAP).fillna(k_adata.obs[cond_col].astype(str))

# Cell type column
k_ct_col = 'mixed_celltype' if 'mixed_celltype' in k_adata.obs.columns else 'celltype'
print(f"Kidney: {k_adata.shape}, ct col: {k_ct_col}")
print(f"Cell types: {list(k_adata.obs[k_ct_col].unique())}")
print(f"Conditions: {list(k_adata.obs['condition'].unique())}")

# ── Cell 5: Define marker sets ───────────────────────────────
# Xu et al. 2024 / Kann et al. 2025 failed-repair PT markers
failed_repair_markers = [
    'Vcam1',      # THE key failed-repair marker
    'Havcr1',     # KIM-1, acute injury (distinct from failed-repair)
    'Krt8',       # Transitional/injured PT
    'Krt18',      # Co-expressed with Krt8
    'Cd44',       # Injury/dedifferentiation
    'Sox9',       # Dedifferentiation/regeneration
    'Tp53',       # p53, damage response
    'Cdkn1a',     # p21, senescence
    'Ccl2',       # Inflammatory, failed-repair
    'Tgfb1',      # Pro-fibrotic
    'Fn1',        # Fibronectin, ECM/injury
    'Vim',        # Vimentin, mesenchymal
    'Col1a1',     # Fibrotic
    'Lgals3',     # Galectin-3, injury/fibrosis
]

# Healthy PT markers (HIGH in PT-1, LOW in PT-2/failed-repair)
healthy_pt_markers = [
    'Slc34a1',    # Sodium-phosphate cotransporter
    'Lrp2',       # Megalin
    'Slc5a2',     # SGLT2
    'Slc22a6',    # OAT1
    'Slc22a8',    # OAT3
    'Aqp1',       # Aquaporin 1
    'Cubn',       # Cubilin
    'Slc7a13',    # PT-specific transporter
]

all_markers = failed_repair_markers + healthy_pt_markers
available = [g for g in all_markers if g in k_adata.var_names]
missing = [g for g in all_markers if g not in k_adata.var_names]

fr_avail = [g for g in failed_repair_markers if g in available]
hp_avail = [g for g in healthy_pt_markers if g in available]

print(f"Available: {len(available)}/{len(all_markers)}")
print(f"Missing: {missing}")
print(f"\nFailed-repair markers: {fr_avail}")
print(f"Healthy PT markers: {hp_avail}")

# ── Cell 6: DotPlot — PT-1 vs PT-2 per condition ─────────────
k_pt = k_adata[k_adata.obs[k_ct_col].isin(['PT-1', 'PT-2'])].copy()
k_pt.obs['ct_cond'] = k_pt.obs[k_ct_col].astype(str) + '_' + k_pt.obs['condition'].astype(str)

ct_cond_order = [f'{ct}_{c}' for ct in ['PT-1', 'PT-2'] for c in cond_order]
ct_cond_order = [x for x in ct_cond_order if x in k_pt.obs['ct_cond'].unique()]

marker_dict = {}
if fr_avail:
    marker_dict['Failed-repair'] = fr_avail
if hp_avail:
    marker_dict['Healthy PT'] = hp_avail

sc.pl.dotplot(k_pt, var_names=marker_dict, groupby='ct_cond',
              categories_order=ct_cond_order, standard_scale='var',
              show=False, figsize=(16, 6))
plt.title('PT-1 vs PT-2: Xu/Kann failed-repair markers vs healthy PT markers')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}20_PT_failed_repair_reannotation.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 20_PT_failed_repair_reannotation.png")

# ── Cell 7: DotPlot — All kidney cell types (context) ────────
sc.pl.dotplot(k_adata, var_names=marker_dict, groupby=k_ct_col,
              standard_scale='var', show=False, figsize=(16, 8))
plt.title('Failed-repair vs healthy PT markers across all kidney cell types')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}21_PT_failed_repair_all_celltypes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: 21_PT_failed_repair_all_celltypes.png")

# ── Cell 8: Mean expression CSV ──────────────────────────────
expr_data = pd.DataFrame()
for ct in ['PT-1', 'PT-2']:
    for cond in cond_order:
        mask = (k_adata.obs[k_ct_col] == ct) & (k_adata.obs['condition'] == cond)
        subset = k_adata[mask]
        if subset.shape[0] > 0:
            avail_here = [g for g in available if g in subset.var_names]
            if hasattr(subset.X, 'toarray'):
                means = pd.DataFrame(subset[:, avail_here].X.toarray(),
                                     columns=avail_here).mean()
            else:
                means = pd.DataFrame(subset[:, avail_here].X,
                                     columns=avail_here).mean()
            means['celltype'] = ct
            means['condition'] = cond
            means['n_cells'] = subset.shape[0]
            expr_data = pd.concat([expr_data, means.to_frame().T], ignore_index=True)

expr_data.to_csv(f'{OUTPUT_DIR}20_PT_failed_repair_expression.csv', index=False)
print("Saved: 20_PT_failed_repair_expression.csv")

# Print key comparisons
print("\n" + "="*60)
print("KEY COMPARISONS: PT-1 vs PT-2")
print("="*60)
for marker in ['Vcam1', 'Havcr1', 'Krt8', 'Cd44', 'Slc34a1', 'Lrp2']:
    if marker in expr_data.columns:
        print(f"\n{marker}:")
        for ct in ['PT-1', 'PT-2']:
            vals = expr_data[expr_data['celltype'] == ct].set_index('condition')
            if marker in vals.columns:
                for cond in cond_order:
                    if cond in vals.index:
                        print(f"  {ct}_{cond}: {float(vals.loc[cond, marker]):.4f}")

# ── Cell 9: Detection rates (% cells expressing) ─────────────
frac_data = pd.DataFrame()
key_markers = [g for g in ['Vcam1', 'Havcr1', 'Krt8', 'Krt18', 'Cd44',
                           'Cdkn1a', 'Slc34a1', 'Lrp2', 'Aqp1'] if g in available]

for ct in ['PT-1', 'PT-2']:
    for cond in cond_order:
        mask = (k_adata.obs[k_ct_col] == ct) & (k_adata.obs['condition'] == cond)
        subset = k_adata[mask]
        if subset.shape[0] > 0:
            row = {'celltype': ct, 'condition': cond, 'n_cells': subset.shape[0]}
            for marker in key_markers:
                if marker in subset.var_names:
                    if hasattr(subset.X, 'toarray'):
                        vals = subset[:, marker].X.toarray().flatten()
                    else:
                        vals = subset[:, marker].X.flatten()
                    row[f'{marker}_pct'] = (vals > 0).sum() / len(vals) * 100
                    row[f'{marker}_mean'] = vals.mean()
            frac_data = pd.concat([frac_data, pd.DataFrame([row])], ignore_index=True)

frac_data.to_csv(f'{OUTPUT_DIR}22_PT_marker_detection_rates.csv', index=False)
print("Saved: 22_PT_marker_detection_rates.csv")

print("\n" + "="*60)
print("DETECTION RATES (% cells expressing > 0)")
print("="*60)
for marker in key_markers:
    pct_col = f'{marker}_pct'
    if pct_col in frac_data.columns:
        print(f"\n{marker}:")
        for ct in ['PT-1', 'PT-2']:
            ct_data = frac_data[frac_data['celltype'] == ct].set_index('condition')
            for cond in cond_order:
                if cond in ct_data.index:
                    pct = float(ct_data.loc[cond, pct_col])
                    mean = float(ct_data.loc[cond, f'{marker}_mean'])
                    print(f"  {ct}_{cond}: {pct:.1f}% (mean={mean:.4f})")

# ── Cell 10: Summary ─────────────────────────────────────────
print("\n" + "="*60)
print("INTERPRETATION GUIDE")
print("="*60)
print("""
If PT-2 shows:
  HIGH Vcam1 + HIGH Krt8 + LOW Slc34a1/Lrp2
  → PT-2 = failed-repair PT (Xu 2024 / Kann 2025)

If PT-2 shows:
  HIGH Havcr1 but LOW Vcam1
  → PT-2 = acutely injured, not yet committed to failed-repair

Key finding for Discussion:
  Combi lowers Vcam1 (failed-repair) but NOT Havcr1 (acute injury)
  → intervention shifts injured tubules OFF the failed-repair trajectory
     without restoring healthy tubule identity
""")

print("\nAll outputs saved to:")
print(f"  {OUTPUT_DIR}")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.startswith(('20', '21', '22')):
        size = os.path.getsize(f'{OUTPUT_DIR}{f}')
        print(f"  {f} ({size/1024:.1f} KB)")

# Immune cell types check

In [ ]:
#!/usr/bin/env python3
"""
Comprehensive immune marker check — both tissues
Checks ALL markers from colleague's list + detection rates + composition shifts
Colab version — mount Drive first.
"""

# ── Cell 1: Mount Drive ──────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Cell 2: Setup ────────────────────────────────────────────
!pip install scanpy anndata -q

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

SN_DIR = "/content/drive/MyDrive/IMSB/projects/Hubers lab/Aging_DEG_Enrichment_Cellchat/files/sn"
OUTPUT_DIR = "/content/drive/MyDrive/IMSB/projects/Hubers lab/Aging_DEG_Enrichment_Cellchat/thesis_supp_analysis/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

KIDNEY_CONDITION_MAP = {
    'sg18': 'ctrl', 'sg20': 'age', 'sg24': 'DR', 'sg26': 'sAct', 'sg28': 'combi'
}
MUSCLE_CONDITION_MAP = {
    'rgzj1': 'ctrl', 'rgzj2': 'age', 'rgzj3': 'DR', 'rgzj4': 'sAct', 'rgzj5': 'combi'
}
cond_order = ['ctrl', 'age', 'DR', 'sAct', 'combi']

# ── Cell 3: Full marker list from colleague ──────────────────
# Organized by lineage
immune_markers_full = {
    'Macrophage/Myeloid': ['Cd68', 'Adgre1', 'Csf1r', 'Cd14', 'Fcgr3'],
    'MHC-II/Antigen pres.': ['H2-Aa', 'H2-Ab1', 'H2-Eb1'],
    'Dendritic': ['Cd80', 'Clec9a', 'Itgax'],
    'T cell': ['Cd3d', 'Cd3e', 'Cd8a', 'Cd4'],
    'NK cell': ['Ncr1', 'Klrb1c'],
    'B cell': ['Cd19', 'Ms4a1', 'Mzb1'],
    'Neutrophil': ['Ly6g', 'S100a8', 'S100a9'],
    'Mast cell': ['Kit', 'Cpa3'],
    'Monocyte CD16': ['Fcgr3'],  # already in macrophage list
    'General immune': ['Ptprc'],  # CD45
}

all_markers = list(set(g for genes in immune_markers_full.values() for g in genes))
all_markers.append('Ptprc')  # CD45 - pan-leukocyte
all_markers = sorted(set(all_markers))

# ── Cell 4: Load kidney ──────────────────────────────────────
print("Loading kidney...")
k_adata = sc.read_h5ad(f"{SN_DIR}/annotated-aging-soupxed.h5ad")
if 'sample' in k_adata.obs.columns:
    cond_col = 'sample'
elif 'orig.ident' in k_adata.obs.columns:
    cond_col = 'orig.ident'
else:
    cond_col = k_adata.obs.columns[0]
k_adata.obs['condition'] = k_adata.obs[cond_col].map(KIDNEY_CONDITION_MAP).fillna(k_adata.obs[cond_col])
k_ct_col = 'mixed_celltype' if 'mixed_celltype' in k_adata.obs.columns else 'celltype'
print(f"Kidney: {k_adata.shape}")

# ── Cell 5: Load muscle ──────────────────────────────────────
print("Loading muscle...")
m_adata = sc.read_h5ad(f"{SN_DIR}/annotated-muscle-soupxed.h5ad")
if 'sample' in m_adata.obs.columns:
    m_cond_col = 'sample'
elif 'orig.ident' in m_adata.obs.columns:
    m_cond_col = 'orig.ident'
else:
    m_cond_col = m_adata.obs.columns[0]
m_adata.obs['condition'] = m_adata.obs[m_cond_col].astype(str).map(MUSCLE_CONDITION_MAP).fillna(m_adata.obs[m_cond_col].astype(str))
m_ct_col = 'mixed_celltype' if 'mixed_celltype' in m_adata.obs.columns else 'celltype'
print(f"Muscle: {m_adata.shape}")

# ── Cell 6: Availability check ───────────────────────────────
print("\n" + "="*60)
print("MARKER AVAILABILITY")
print("="*60)

for tissue, adata in [('Kidney', k_adata), ('Muscle', m_adata)]:
    print(f"\n--- {tissue} ---")
    for category, genes in immune_markers_full.items():
        avail = [g for g in genes if g in adata.var_names]
        miss = [g for g in genes if g not in adata.var_names]
        status = "✅" if len(miss) == 0 else "⚠️" if len(avail) > 0 else "❌"
        print(f"  {status} {category}: {len(avail)}/{len(genes)} available")
        if miss:
            print(f"      Missing: {miss}")

# ── Cell 7: Kidney IMM — comprehensive detection rates ───────
print("\n" + "="*60)
print("KIDNEY IMM: Detection rates across conditions")
print("="*60)

available_k = [g for g in all_markers if g in k_adata.var_names]
k_imm = k_adata[k_adata.obs[k_ct_col] == 'IMM'].copy()

det_data_k = pd.DataFrame()
for cond in cond_order:
    mask = k_imm.obs['condition'] == cond
    subset = k_imm[mask]
    if subset.shape[0] > 0:
        row = {'condition': cond, 'n_cells': subset.shape[0]}
        for marker in available_k:
            if marker in subset.var_names:
                if hasattr(subset.X, 'toarray'):
                    vals = subset[:, marker].X.toarray().flatten()
                else:
                    vals = subset[:, marker].X.flatten()
                row[f'{marker}_pct'] = (vals > 0).sum() / len(vals) * 100
                row[f'{marker}_mean'] = vals.mean()
        det_data_k = pd.concat([det_data_k, pd.DataFrame([row])], ignore_index=True)

det_data_k.to_csv(f'{OUTPUT_DIR}23_kidney_IMM_comprehensive_detection.csv', index=False)
print("Saved: 23_kidney_IMM_comprehensive_detection.csv")

# Print key results
for marker in available_k:
    pct_col = f'{marker}_pct'
    if pct_col in det_data_k.columns:
        vals = det_data_k.set_index('condition')[pct_col]
        if vals.max() > 1.0:  # only show markers with >1% detection in any condition
            print(f"\n{marker}:")
            for cond in cond_order:
                if cond in vals.index:
                    pct = float(vals[cond])
                    mean = float(det_data_k.set_index('condition').loc[cond, f'{marker}_mean'])
                    print(f"  {cond}: {pct:.1f}% (mean={mean:.3f})")

# ── Cell 8: Kidney — immune markers across ALL cell types ────
print("\n" + "="*60)
print("KIDNEY: Immune markers across all cell types (detection %)")
print("="*60)

# Check which cell types express immune markers
ct_immune_k = pd.DataFrame()
for ct in sorted(k_adata.obs[k_ct_col].unique()):
    subset = k_adata[k_adata.obs[k_ct_col] == ct]
    row = {'celltype': ct, 'n_cells': subset.shape[0]}
    for marker in ['Ptprc', 'Adgre1', 'Csf1r', 'Cd68', 'H2-Aa', 'Ms4a1', 'Kit']:
        if marker in available_k:
            if hasattr(subset.X, 'toarray'):
                vals = subset[:, marker].X.toarray().flatten()
            else:
                vals = subset[:, marker].X.flatten()
            row[f'{marker}_pct'] = (vals > 0).sum() / len(vals) * 100
    ct_immune_k = pd.concat([ct_immune_k, pd.DataFrame([row])], ignore_index=True)

ct_immune_k.to_csv(f'{OUTPUT_DIR}24_kidney_immune_by_celltype.csv', index=False)
print("Saved: 24_kidney_immune_by_celltype.csv")

# Print cell types with >5% Ptprc (CD45) or Adgre1 detection
print("\nCell types with >5% Ptprc (CD45) or Adgre1 detection:")
for _, row in ct_immune_k.iterrows():
    ptprc = row.get('Ptprc_pct', 0)
    adgre = row.get('Adgre1_pct', 0)
    if ptprc > 5 or adgre > 5:
        print(f"  {row['celltype']}: Ptprc={ptprc:.1f}%, Adgre1={adgre:.1f}%")

# ── Cell 9: Muscle — immune markers in macrophages + immune cells ──
print("\n" + "="*60)
print("MUSCLE: Immune detection in macrophages/immune cells")
print("="*60)

available_m = [g for g in all_markers if g in m_adata.var_names]

# Check macrophages and immune cells?
for ct_name in ['Macrophages', 'Immune cells?']:
    if ct_name in m_adata.obs[m_ct_col].unique():
        m_ct = m_adata[m_adata.obs[m_ct_col] == ct_name].copy()
        print(f"\n--- {ct_name} ---")

        det_data_m = pd.DataFrame()
        for cond in cond_order:
            mask = m_ct.obs['condition'] == cond
            subset = m_ct[mask]
            if subset.shape[0] > 0:
                row = {'condition': cond, 'n_cells': subset.shape[0]}
                for marker in available_m:
                    if marker in subset.var_names:
                        if hasattr(subset.X, 'toarray'):
                            vals = subset[:, marker].X.toarray().flatten()
                        else:
                            vals = subset[:, marker].X.flatten()
                        row[f'{marker}_pct'] = (vals > 0).sum() / len(vals) * 100
                        row[f'{marker}_mean'] = vals.mean()
                det_data_m = pd.concat([det_data_m, pd.DataFrame([row])], ignore_index=True)

        ct_safe = ct_name.replace('?', '').replace(' ', '_')
        det_data_m.to_csv(f'{OUTPUT_DIR}25_muscle_{ct_safe}_detection.csv', index=False)
        print(f"Saved: 25_muscle_{ct_safe}_detection.csv")

        for marker in available_m:
            pct_col = f'{marker}_pct'
            if pct_col in det_data_m.columns:
                vals = det_data_m.set_index('condition')[pct_col]
                if vals.max() > 1.0:
                    print(f"\n  {marker}:")
                    for cond in cond_order:
                        if cond in vals.index:
                            print(f"    {cond}: {float(vals[cond]):.1f}%")

# ── Cell 10: DotPlot — kidney IMM comprehensive ──────────────
# Group available markers by category for a clean DotPlot
imm_dict_k = {}
for cat, genes in immune_markers_full.items():
    avail = [g for g in genes if g in k_adata.var_names]
    if avail:
        imm_dict_k[cat] = avail

if imm_dict_k:
    k_imm_plot = k_adata[k_adata.obs[k_ct_col] == 'IMM'].copy()
    sc.pl.dotplot(k_imm_plot, var_names=imm_dict_k, groupby='condition',
                  categories_order=cond_order, standard_scale='var',
                  show=False, figsize=(16, 5))
    plt.title('Kidney IMM: comprehensive immune marker panel across conditions')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}23_kidney_IMM_comprehensive_dotplot.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: 23_kidney_IMM_comprehensive_dotplot.png")

# ── Cell 11: IMM composition shift ───────────────────────────
print("\n" + "="*60)
print("IMMUNE CELL COMPOSITION SHIFTS")
print("="*60)

# Kidney IMM nuclei counts
print("\nKidney IMM nuclei per condition:")
for cond in cond_order:
    n = (k_adata.obs[k_ct_col] == 'IMM') & (k_adata.obs['condition'] == cond)
    total = (k_adata.obs['condition'] == cond).sum()
    count = n.sum()
    pct = count / total * 100
    print(f"  {cond}: {count} nuclei ({pct:.1f}%)")

# Muscle macrophage + immune counts
print("\nMuscle Macrophages nuclei per condition:")
for cond in cond_order:
    for ct_name in ['Macrophages', 'Immune cells?']:
        if ct_name in m_adata.obs[m_ct_col].unique():
            n = (m_adata.obs[m_ct_col] == ct_name) & (m_adata.obs['condition'] == cond)
            total = (m_adata.obs['condition'] == cond).sum()
            count = n.sum()
            pct = count / total * 100
            print(f"  {cond} [{ct_name}]: {count} nuclei ({pct:.1f}%)")

# ── Cell 12: Summary ─────────────────────────────────────────
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print("""
Key findings to check:
1. Which immune markers show condition-dependent changes?
2. Does MHC-II (H2-Aa/Ab1/Eb1) decrease with aging AND combi? (already confirmed)
3. Does Ptprc (CD45) detection change across conditions?
4. Any T/B/NK signal at all? (likely no — snRNA limitation)
5. Does macrophage composition (% of total) shift across conditions?
6. Any signal in the 'Immune cells?' cluster in muscle?
""")

for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.startswith(('23', '24', '25')):
        size = os.path.getsize(f'{OUTPUT_DIR}{f}')
        print(f"  {f} ({size/1024:.1f} KB)")

### Integrating Claude API

To use the Claude API, you'll need an API key from Anthropic. If you don't have one, you can obtain it from their website.

In Colab, it's best practice to store your API key securely in the secrets manager. You can find this under the "🔑" icon in the left panel. Name the secret `ANTHROPIC_API_KEY`.

In [ ]:
# First, install the Anthropic Python client library
!pip install anthropic

# Import necessary libraries
import anthropic
from google.colab import userdata

# Retrieve your API key from Colab secrets
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

# Initialize the Claude client
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

Now you can make API calls to Claude. Here's an example of how to generate text using the `messages.create` method with the latest available model.

In [ ]:
try:
    message = client.messages.create(
        model="claude-3-opus-20240229", # Or another suitable Claude 3 model
        max_tokens=1024,
        messages=[
            {"role": "user", "content": "Write a short story about a brave mouse."}
        ]
    )
    print(message.content[0].text)
except Exception as e:
    print(f"An error occurred: {e}")